In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
sys.path.append('..')
from src import functions as fc

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [4]:
df_modeling = fc.load_data_clean("bank_final.csv")

Buscando archivo en: /Users/jbp/Desktop/IRONHACK/SEMANA7/ML_project/data/cleaned/bank_final.csv


In [5]:
df_modeling.shape

(4928, 64)

In [6]:
features = df_modeling.drop(columns=["target", "duration"])
target = df_modeling["target"]

In [7]:
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=0)

Random Forest:

In [8]:
forest = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

Hemos reducido el max depth a 10 para que no memorice los datos y no haya overfitting. 

In [9]:
forest.fit(x_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
pred = forest.predict(x_test)

In [11]:
accuracy = forest.score(x_test, y_test)
print(f"La precisión del modelo es: {accuracy:.2f}")

La precisión del modelo es: 0.64


In [12]:
importancias = pd.DataFrame({'feature': features.columns, 'importance': forest.feature_importances_}).sort_values('importance', ascending=False)
print(importancias.head(10))

              feature  importance
7           euribor3m    0.132353
0                 age    0.083121
8         nr.employed    0.065616
1            campaign    0.049538
4        emp.var.rate    0.045846
5      cons.price.idx    0.043288
6       cons.conf.idx    0.036460
2               pdays    0.030590
42   contact_cellular    0.022243
43  contact_telephone    0.020492


Random Search:

In [13]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist_forest = {
    'n_estimators': randint(200, 1000),
    'max_depth': [None] + list(range(5, 40, 5)),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced', 'balanced_subsample']}

random_forest = RandomizedSearchCV(
    estimator=RandomForestClassifier(),
    param_distributions=param_dist_forest,
    n_iter=80,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=2,
    random_state=42)

random_forest.fit(x_train, y_train)

Fitting 5 folds for each of 80 candidates, totalling 400 fits
[CV] END bootstrap=True, class_weight=None, max_depth=30, max_features=None, min_samples_leaf=8, min_samples_split=8, n_estimators=321; total time=   2.8s
[CV] END bootstrap=True, class_weight=None, max_depth=30, max_features=None, min_samples_leaf=8, min_samples_split=8, n_estimators=321; total time=   2.8s
[CV] END bootstrap=True, class_weight=None, max_depth=30, max_features=None, min_samples_leaf=8, min_samples_split=8, n_estimators=321; total time=   2.9s
[CV] END bootstrap=True, class_weight=None, max_depth=30, max_features=None, min_samples_leaf=8, min_samples_split=8, n_estimators=321; total time=   2.9s
[CV] END bootstrap=True, class_weight=None, max_depth=30, max_features=None, min_samples_leaf=8, min_samples_split=8, n_estimators=321; total time=   3.0s
[CV] END bootstrap=True, class_weight=balanced, max_depth=20, max_features=log2, min_samples_leaf=8, min_samples_split=13, n_estimators=613; total time=   1.0s
[CV

,estimator,RandomForestClassifier()
,param_distributions,"{'bootstrap': [True, False], 'class_weight': [None, 'balanced', ...], 'max_depth': [None, 5, ...], 'max_features': ['sqrt', 'log2', ...], ...}"
,n_iter,80
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [14]:
random_forest.best_params_

{'bootstrap': False,
 'class_weight': 'balanced_subsample',
 'max_depth': 25,
 'max_features': 'log2',
 'min_samples_leaf': 3,
 'min_samples_split': 13,
 'n_estimators': 766}

In [15]:
best_model2 = random_forest.best_estimator_

In [16]:
accuracy2 = best_model2.score(x_test, y_test)
print(f"La precisión del modelo es: {accuracy2:.2f}")

La precisión del modelo es: 0.60


In [17]:
from sklearn.metrics import precision_score, recall_score, f1_score

pred2 = best_model2.predict(x_test)

precision2 = precision_score(y_test, pred2)
recall2 = recall_score(y_test, pred2)
f12 = f1_score(y_test, pred2)

print(f"Precision del modelo es: {precision2:.2f}")
print(f"Recall del modelo es: {recall2:.2f}")
print(f"F1-score del modelo es: {f12:.2f}")

Precision del modelo es: 0.51
Recall del modelo es: 0.51
F1-score del modelo es: 0.51


El dataset presenta una distribución relativamente equilibrada entre ambas clases, con un 58% de clientes que no contrataron y un 42% que sí contrataron.

Por este motivo, la accuracy es la única métrica válida para evaluar el rendimiento general del modelo.

Dado que el objetivo del proyecto es identificar clientes que contratarán el depósito, es especialmente importante analizar métricas como recall y F1-score.

El recall mide la capacidad del modelo para detectar clientes que realmente contratarán el producto, mientras que el F1-score proporciona una medida equilibrada entre precision y recall.

El modelo Random Forest obtuvo el mayor F1-score (0.52) y el mayor recall (0.53), lo que indica que es el modelo más eficaz para identificar clientes potenciales.

Por este motivo, Random Forest fue seleccionado como modelo final.

In [20]:
import joblib

joblib.dump(best_model2, "../streamlit_app/random_forest_model.pkl")

['../streamlit_app/random_forest_model.pkl']

In [30]:
# crear dataframe test completo

test_df = x_test.copy()
test_df["real"] = y_test

test_df["pred"] = best_model2.predict(x_test)

In [31]:
correct_no = test_df[(test_df["pred"] == 0) & (test_df["real"] == 0)].iloc[0]

correct_yes = test_df[(test_df["pred"] == 1) & (test_df["real"] == 1)].iloc[0]


print("CASO CORRECTO NO")
print(correct_no[['age','campaign','previous','euribor3m','nr.employed','emp.var.rate']])

print("\nCASO CORRECTO YES")
print(correct_yes[['age','campaign','previous','euribor3m','nr.employed','emp.var.rate']])


print("\nVERIFICACIÓN:")

print("\nNO caso:")
print("Pred:", correct_no["pred"])
print("Real:", correct_no["real"])

print("\nYES caso:")
print("Pred:", correct_yes["pred"])
print("Real:", correct_yes["real"])

CASO CORRECTO NO
age               30.00
campaign           1.00
previous           0.00
euribor3m          4.12
nr.employed     5195.80
emp.var.rate      -0.10
Name: 2946, dtype: float64

CASO CORRECTO YES
age               59.00
campaign           1.00
previous           2.00
euribor3m          0.72
nr.employed     4991.60
emp.var.rate      -1.70
Name: 4727, dtype: float64

VERIFICACIÓN:

NO caso:
Pred: 0.0
Real: 0.0

YES caso:
Pred: 1.0
Real: 1.0
